# Week 3 – Executable PDF/TXT Data Pipeline
**Project:** AI-Powered Contract and Terms & Conditions Analyzer

The supplied examples are synthetic teaching files, not real contracts. This notebook demonstrates ingestion and preparation only; it does not measure contract-risk detection accuracy.

Run from the repository root. Install dependencies with `pip install pymupdf`.

In [1]:
from pathlib import Path
import re, json, hashlib
import fitz

ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw'
OUT = ROOT / 'data' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)
FILES = [RAW/'student_handbook_demo.pdf', RAW/'faq_demo.txt']
print('Input files:', {p.name:p.exists() for p in FILES})

Input files: {'student_handbook_demo.pdf': True, 'faq_demo.txt': True}


In [2]:
def read_doc(path):
    if path.suffix.lower()=='.pdf':
        with fitz.open(path) as pdf:
            return '\n'.join(page.get_text('text') for page in pdf)
    if path.suffix.lower()=='.txt':
        return path.read_text(encoding='utf-8')
    raise ValueError(f'Unsupported format: {path.suffix}')

inspection = {}
for p in FILES:
    if not p.exists(): raise FileNotFoundError(p)
    raw = read_doc(p)
    inspection[p.name] = len(raw)
print(inspection)

{'student_handbook_demo.pdf': 426, 'faq_demo.txt': 171}

In [3]:
def clean(text):
    text=text.replace('\r\n','\n').replace('\r','\n')
    text=re.sub(r'[ \t]+',' ',text)
    text=re.sub(r'\n{3,}','\n\n',text)
    return text.strip()

def structural_chunks(text, filename):
    if filename.endswith('.pdf'):
        chunks=[]; current=[]
        for line in [x.strip() for x in text.splitlines() if x.strip()]:
            if re.match(r'^Section\s+\d+:',line,re.I) and current:
                chunks.append('\n'.join(current)); current=[line]
            else: current.append(line)
        if current: chunks.append('\n'.join(current))
        return chunks
    return [b.strip() for b in re.split(r'\n\s*\n',text.strip()) if b.strip()]

def fixed_chunks(text,size=160):
    return [text[i:i+size].strip() for i in range(0,len(text),size) if text[i:i+size].strip()]

prepared={}; comparison=[]
for p in FILES:
    cleaned=clean(read_doc(p)); chunks=structural_chunks(cleaned,p.name)
    prepared[p.name]=(cleaned,chunks)
    comparison.append((p.name,len(chunks),len(fixed_chunks(cleaned))))
print(comparison)

[('student_handbook_demo.pdf', 5, 3), ('faq_demo.txt', 2, 2)]

In [4]:
validation=[]
for p in FILES:
    cleaned,chunks=prepared[p.name]
    assert cleaned and chunks
    digest=hashlib.sha256(p.read_bytes()).hexdigest()
    target=OUT/f'{p.stem}_prepared.json'
    target.write_text(json.dumps({'source_file':p.name,'source_sha256':digest,'chunks':chunks},indent=2),encoding='utf-8')
    check=json.loads(target.read_text(encoding='utf-8'))
    assert check['chunks']==chunks
    validation.append({'source':p.name,'saved':True,'chunks':len(chunks)})
print(validation)

[{'source': 'student_handbook_demo.pdf', 'saved': True, 'chunks': 5}, {'source': 'faq_demo.txt', 'saved': True, 'chunks': 2}]

In [5]:
repeatability={}
for p in FILES:
    first=(clean(read_doc(p)),structural_chunks(clean(read_doc(p)),p.name))
    second=(clean(read_doc(p)),structural_chunks(clean(read_doc(p)),p.name))
    repeatability[p.name]=(first==second)
    assert first==second
print(repeatability)

{'student_handbook_demo.pdf': True, 'faq_demo.txt': True}

## Decisions, failure points and limitations

**Selected method:** structure-aware chunking. It preserves PDF section headings and TXT FAQ blocks. Fixed 160-character chunking is retained as a baseline and may split semantic units.

**Potential failures:** (1) scanned PDFs may return empty text—detect and report OCR requirement; (2) missing/unsupported files—validate path and extension; (3) invalid TXT encoding—surface the decoding error.

**Limitations:** Synthetic examples only; no contract clauses, labels, model training or legal-risk evaluation. OCR, password-protected PDFs and multilingual extraction are outside this V1 pipeline.